In [14]:
import psycopg2
import json
from decimal import Decimal
from datetime import date, datetime


def connect_db():
    try:
        conn = psycopg2.connect(
            dbname="Renewly",
            user="ayushmayekar",   # If you get "role postgres does not exist", use your system username or create the role
            password="password",
            host="localhost",
            port="5432"
        )
        print(" Connected to PostgreSQL successfully")
        return conn
    except Exception as e:
        print(" Connection failed:", e)
        raise RuntimeError("Cannot connect to DB. Fix credentials or start PostgreSQL.") from e

In [15]:
def get_tables_and_columns(conn):
    cursor = conn.cursor()

    # Exclude partition child tables so we get one schema entry per logical table.
    # In PostgreSQL each partition appears as a separate table; we keep only the
    # parent partitioned table (and non-partitioned tables).
    cursor.execute("""
        SELECT table_name, column_name, data_type
        FROM information_schema.columns c
        WHERE c.table_schema = 'public'
          AND c.table_name NOT IN (
              SELECT child.relname
              FROM pg_inherits i
              JOIN pg_class child ON child.oid = i.inhrelid
              JOIN pg_namespace n ON n.oid = child.relnamespace
              WHERE n.nspname = 'public'
          )
        ORDER BY table_name, ordinal_position;
    """)

    rows = cursor.fetchall()

    schema = {}

    for table, column, dtype in rows:
        if table not in schema:
            schema[table] = {
                "columns": [],
                "primary_keys": [],
                "foreign_keys": []
            }

        schema[table]["columns"].append({
            "column_name": column,
            "data_type": dtype
        })

    return schema



def get_primary_keys(conn, schema):
    cursor = conn.cursor()

    cursor.execute("""
        SELECT
            tc.table_name,
            kcu.column_name
        FROM
            information_schema.table_constraints tc
        JOIN
            information_schema.key_column_usage kcu
        ON tc.constraint_name = kcu.constraint_name
        WHERE
            tc.constraint_type = 'PRIMARY KEY'
            AND tc.table_schema = 'public';
    """)

    rows = cursor.fetchall()

    for table, column in rows:
        if table in schema and column not in schema[table]["primary_keys"]:
            schema[table]["primary_keys"].append(column)

    return schema


def get_foreign_keys(conn, schema):
    cursor = conn.cursor()

    cursor.execute("""
        SELECT
            tc.table_name,
            kcu.column_name,
            ccu.table_name AS foreign_table,
            ccu.column_name AS foreign_column
        FROM
            information_schema.table_constraints AS tc
        JOIN information_schema.key_column_usage AS kcu
            ON tc.constraint_name = kcu.constraint_name
        JOIN information_schema.constraint_column_usage AS ccu
            ON ccu.constraint_name = tc.constraint_name
        WHERE tc.constraint_type = 'FOREIGN KEY'
            AND tc.table_schema='public';
    """)

    rows = cursor.fetchall()

    for table, column, f_table, f_column in rows:
        if table not in schema:
            continue
        fk = {
            "column": column,
            "references_table": f_table,
            "references_column": f_column
        }
        existing = schema[table]["foreign_keys"]
        if not any(
            e["column"] == column and e["references_table"] == f_table and e["references_column"] == f_column
            for e in existing
        ):
            existing.append(fk)

    return schema

def mark_indexed_columns(conn, schema):
    cursor = conn.cursor()

    cursor.execute("""
        SELECT
            t.relname AS table_name,
            a.attname AS column_name
        FROM
            pg_class t
        JOIN
            pg_index ix ON t.oid = ix.indrelid
        JOIN
            pg_attribute a ON a.attrelid = t.oid
                AND a.attnum = ANY(ix.indkey)
        WHERE
            t.relkind = 'r';
    """)

    indexed_columns = cursor.fetchall()

    indexed_set = set(indexed_columns)

    for table in schema:
        for column in schema[table]["columns"]:
            col_name = column["column_name"]

            if (table, col_name) in indexed_set:
                column["is_indexed"] = True
            else:
                column["is_indexed"] = False

    return schema

def attach_sample_rows(conn, schema, limit=2):
    cursor = conn.cursor()

    for table, info in schema.items():
        columns = [c["column_name"] for c in info["columns"]]
        if not columns:
            info["sample_rows"] = []
            continue

        col_list_sql = ", ".join(f'"{c}"' for c in columns)
        query = f'SELECT {col_list_sql} FROM "{table}" LIMIT %s'

        try:
            cursor.execute(query, (limit,))
            rows = cursor.fetchall()
        except Exception:
            info["sample_rows"] = []
            continue

        samples = []
        for row in rows:
            sample_row = {}
            for i, col in enumerate(columns):
                val = row[i]
                if isinstance(val, Decimal):
                    val = float(val)
                elif isinstance(val, (datetime, date)):
                    # ISO 8601 string for JSON
                    val = val.isoformat()
                sample_row[col] = val
            samples.append(sample_row)

        info["sample_rows"] = samples

    return schema


def load_full_schema():
    conn = connect_db()

    schema = get_tables_and_columns(conn)
    schema = get_primary_keys(conn, schema)
    schema = get_foreign_keys(conn, schema)
    schema = mark_indexed_columns(conn, schema)
    schema = attach_sample_rows(conn, schema)

    conn.close()
    return schema

In [16]:
def serialize_schema(schema):
    serialized = ""
    table_index_map = {}

    for t_index, (table, info) in enumerate(schema.items()):
        table_alias = f"t{t_index}"
        table_index_map[table] = table_alias

        serialized += f"{table_alias}: {table} ("

        for c_index, column in enumerate(info["columns"]):
            serialized += f"c{c_index}: {column['column_name']}"
            if c_index != len(info["columns"]) - 1:
                serialized += ", "

        serialized += ")\n"

    # Add foreign key relations
    for table, info in schema.items():
        for fk in info["foreign_keys"]:
            serialized += (
                f"{table_index_map[table]}.{fk['column']} = "
                f"{table_index_map[fk['references_table']]}.{fk['references_column']}\n"
            )

    return serialized

In [17]:
schema = load_full_schema()

with open("schema.json", "w") as f:
    json.dump(schema, f, indent=4)

 Connected to PostgreSQL successfully
